# Attention Playground

在这里从零实现并调试 Multi-Head Attention。


In [6]:
import numpy as np
import torch
import torch.nn as nn

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"PyTorch: {torch.__version__} | device: {device}")

PyTorch: 2.10.0 | device: mps


## Decoder-only Transformer


Vanilla Attention


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, p=0.):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(p)
        
    def forward(self, x: torch.Tensor, causal=False):
        """
        x: (batch, seq_len, d_model)
        """
        batch, seq_len = x.size(0), x.size(1)
        print("Input Shape:", x.shape)
        
        # QKV [B, S, d_model] -> [B, S, num_heads, d_head] -> [B,num_heads, S, d_head]
        Q = self.W_Q(x).view(batch, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_K(x).view(batch, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_V(x).view(batch, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # QK^T = score -> [B, nums_heads, S, S]
        score = torch.matmul(Q, K.transpose(-1, -2)) / np.sqrt(self.d_head)
        print("Score Shape:", score.shape)

        # Causal Mask [B, num_heads, S, S]
        if causal:
            causal_mask = torch.triu(torch.ones((seq_len, seq_len),dtype=torch.bool, device=device), diagonal=1)
            score.masked_fill_(causal_mask, float("-inf"))

        # Softmax [B, num_heads, S, S]
        attns = torch.softmax(score, dim = -1)
        attns = self.drop(attns)
        print("Softmax Shape:", attns.shape)

        if causal:
            mask = torch.triu(
                torch.ones(seq_len, seq_len, dtype=torch.bool),
                diagonal=1,
            )
            assert torch.all(attns[..., mask] == 0)

        # output [B, num_heads, S, d_head]
        output = torch.matmul(attns, V)

        # output reshape [B, S, num_heads, d_head] -> [B, S, d_model]
        output = output.transpose(1, 2).contiguous().reshape(batch, seq_len, self.d_model)

        # output [B, S, d_model]
        output = self.W_out(output)
        print("Output Shape:", output.shape)
        return output
            
        

Test Code

In [34]:
B = 2
S = 16
D = 128
H = 8

x = torch.randn(B, S, D, device=device)

mha = MultiHeadAttention(d_model=D, num_heads=H).to(device=device)

output = mha(x, causal=False)
assert output.shape == (B, S, D)

output = mha(x, causal=True)
assert output.shape == (B, S, D)

print("pass")

Input Shape: torch.Size([2, 16, 128])
Score Shape: torch.Size([2, 8, 16, 16])
Softmax Shape: torch.Size([2, 8, 16, 16])
Output Shape: torch.Size([2, 16, 128])
Input Shape: torch.Size([2, 16, 128])
Score Shape: torch.Size([2, 8, 16, 16])
Softmax Shape: torch.Size([2, 8, 16, 16])
Output Shape: torch.Size([2, 16, 128])
pass
